# Movie Recommender System

이 노트북은 tmdb_5000_movies 데이터셋을 활용하여 영화 추천 시스템을 구현합니다.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---
## 1. 데이터 준비 및 이해

In [9]:
# tmdb_5000_movies 데이터를 로드합니다
movies = pd.read_csv('tmdb_5000_movies.csv')

# 데이터의 기본 정보 출력
print("데이터셋의 크기:")
print(f"행: {len(movies)}, 열: {len(movies.columns)}")
print()

# 컬럼명과 데이터 타입
print("\n컬럼명과 데이터 타입:")
print(movies.dtypes)
print()

# 첫 5 개 행 확인
print("\n첫 5 개의 영화 정보:")
print(movies.head())

데이터셋의 크기:
행: 4803, 열: 20


컬럼명과 데이터 타입:
budget                    int64
genres                      str
homepage                    str
id                        int64
keywords                    str
original_language           str
original_title              str
overview                    str
popularity              float64
production_companies        str
production_countries        str
release_date                str
revenue                   int64
runtime                 float64
spoken_languages            str
status                      str
tagline                     str
title                       str
vote_average            float64
vote_count                int64
dtype: object


첫 5 개의 영화 정보:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 

In [ ]:
# 컬럼별 의미 설명
print("컬럼 설명:")
print("- id: 영화 고유 식별자")
print("- title: 영화 제목")
print("- original_title: 원본 제목 (영문)")
print("- overview: 영화 줄거리")
print("- release_date: 개봉 날짜")
print("- genres: 장르 정보 (JSON 형식)")
print("- keywords: 키워드 정보 (JSON 형식)")
print("- budget: 제작비 ($)")
print("- revenue: 수익 ($)")

컬럼 설명:
- movie_id: 영화 고유 식별자
- title: 영화 제목
- original_title: 원본 제목 (영문)
- overview: 영화 줄거리
- release_date: 개봉 날짜
- genres: 장르 정보 (JSON 형식)
- keywords: 키워드 정보 (JSON 형식)
- budget: 제작비 ($)
- revenue: 수익 ($)


---
## 2. 데이터 전처리

`genres`와 `keywords` 칼럼의 데이터를 리스트 형태로 변환합니다.

In [10]:
# genres 컬럼을 리스트 형태로 변환
print("genres 컬럼 샘플 (변환 전):")
print(movies['genres'].head())

print("\ngenres 컬럼 샘플 (변환 후):")
# eval() 을 사용하여 JSON 형식 데이터를 리스트로 변환
movies['genres'] = movies['genres'].apply(eval)

genres 컬럼 샘플 (변환 전):
0    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
1    [{"id": 12, "name": "Adventure"}, {"id": 14, "...
2    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
3    [{"id": 28, "name": "Action"}, {"id": 80, "nam...
4    [{"id": 28, "name": "Action"}, {"id": 12, "nam...
Name: genres, dtype: str

genres 컬럼 샘플 (변환 후):


In [11]:
# keywords 컬럼을 리스트 형태로 변환
print("keywords 컬럼 샘플 (변환 전):")
print(movies['keywords'].head())

print("\nkeywords 컬럼 샘플 (변환 후):")
# eval() 을 사용하여 JSON 형식 데이터를 리스트로 변환
movies['keywords'] = movies['keywords'].apply(eval)

keywords 컬럼 샘플 (변환 전):
0    [{"id": 1463, "name": "culture clash"}, {"id":...
1    [{"id": 270, "name": "ocean"}, {"id": 726, "na...
2    [{"id": 470, "name": "spy"}, {"id": 818, "name...
3    [{"id": 849, "name": "dc comics"}, {"id": 853,...
4    [{"id": 818, "name": "based on novel"}, {"id":...
Name: keywords, dtype: str

keywords 컬럼 샘플 (변환 후):


In [12]:
# 변환 후 데이터 확인
print("변환된 genres 컬럼의 샘플:")
print(movies[['id', 'title', 'genres']].head())

변환된 genres 컬럼의 샘플:
       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  
0  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...  
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...  
2  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...  
3  [{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...  
4  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...  


---
## 3. 피처 엔지니어링

`genres` 칼럼에서 장르 이름만 추출하여 새로운 칼럼 `genres_literal`를 생성합니다.

In [13]:
# genres 컬럼에서 장르 이름 추출
print("genres_literal 컬럼 생성:")

def extract_genres(genre_list):
    """
    장르 리스트에서 장르 이름만 추출하고 공백으로 연결하여 문자열로 변환합니다.
    
    예: [{'name': 'Action', 'id': 28}, {'name': 'Drama', 'id': 18}] -> 'Action Drama'
    """
    # 장르 이름만 추출
    genre_names = [genre['name'] for genre in genre_list]
    # 공백으로 연결하여 문자열로 변환
    return ' '.join(genre_names)

# genres_literal 컬럼 생성
movies['genres_literal'] = movies['genres'].apply(extract_genres)

genres_literal 컬럼 생성:


In [14]:
# 결과 확인
print("genres_literal 컬럼의 샘플:")
print(movies[['id', 'title', 'genres', 'genres_literal']].head())

genres_literal 컬럼의 샘플:
       id                                     title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                              genres  \
0  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...   
3  [{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...   
4  [{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...   

                             genres_literal  
0  Action Adventure Fantasy Science Fiction  
1                  Adventure Fantasy Action  
2                    Action Adventure Crime  
3               Action Crime Drama Thriller  
4          Action Adventure Science Fiction  


---
## 4. 벡터화

`genres_literal` 칼럼을 CountVectorizer 를 활용하여 벡터화합니다.

In [15]:
# CountVectorizer 를 활용한 벡터화
count_vectorizer = CountVectorizer()

# genres_literal를 벡터화
genre_vectors = count_vectorizer.fit_transform(movies['genres_literal'])

In [16]:
# 벡터의 shape 확인
print("벡터의 shape:")
print(f"  행 (영화 개수): {genre_vectors.shape[0]}")
print(f"  열 (고유 장르 수): {genre_vectors.shape[1]}")

# 전체 데이터 개수 확인
print(f"\n전체 데이터 개수: {len(movies)}")

벡터의 shape:
  행 (영화 개수): 4803
  열 (고유 장르 수): 22

전체 데이터 개수: 4803


In [17]:
# genres_literal 을 공백으로 분리한 고유값의 개수 확인
print("\ngenres_literal 컬럼 분석:")

# 공백으로 분리 후 고유값 개수
genres_split = movies['genres_literal'].str.split().apply(lambda x: ' '.join(x))
unique_genres_set = set(''.join(genres_split).split())
print(f"공백으로 분리한 고유 장르의 개수: {len(unique_genres_set)}")

# 고유 장르 목록 확인 (일부)
unique_genres_list = sorted(list(unique_genres_set))
print(f"\n고유 장르 목록 ({len(unique_genres_list)}개): {unique_genres_list}")


genres_literal 컬럼 분석:
공백으로 분리한 고유 장르의 개수: 726

고유 장르 목록 (726개): ['Action', 'ActionAction', 'ActionAdventure', 'ActionAnimation', 'ActionComedy', 'ActionComedyAnimation', 'ActionComedyDrama', 'ActionCrime', 'ActionDocumentaryHistory', 'ActionDrama', 'ActionDramaAction', 'ActionDramaAdventure', 'ActionDramaDrama', 'ActionDramaDramaFantasy', 'ActionFamily', 'ActionFantasy', 'ActionHorror', 'ActionHorrorHorror', 'ActionMusic', 'ActionRomance', 'ActionScience', 'ActionThriller', 'ActionThrillerComedyComedy', 'ActionThrillerHorror', 'ActionThrillerMystery', 'ActionThrillerThriller', 'ActionWar', 'Adventure', 'AdventureAction', 'AdventureAdventure', 'AdventureAdventureFamily', 'AdventureAnimation', 'AdventureComedy', 'AdventureComedyAction', 'AdventureComedyAnimation', 'AdventureComedyComedy', 'AdventureComedyComedyComedy', 'AdventureCrime', 'AdventureDrama', 'AdventureDramaAction', 'AdventureDramaDrama', 'AdventureDramaDramaAction', 'AdventureDramaHorror', 'AdventureDramaRomance', 'Adventur

In [18]:
# 벡터화 결과 확인 (일부 영화)
print("\n벡터화된 데이터 (첫 5 개의 영화):")
print(f"  장르 수: {genre_vectors.shape[1]}")
print()
# 각 영화의 장르 벡터 확인
for i in range(5):
    print(f"\n영화 {i+1}: {movies.iloc[i]['title']}")
    print(f"  장르: {movies.iloc[i]['genres_literal']}")
    print(f"  벡터 형태:")
    print(f"    dtype: {genre_vectors.dtype}")
    print(f"    sparse shape: {genre_vectors[i].shape}")


벡터화된 데이터 (첫 5 개의 영화):
  장르 수: 22


영화 1: Avatar
  장르: Action Adventure Fantasy Science Fiction
  벡터 형태:
    dtype: int64
    sparse shape: (1, 22)

영화 2: Pirates of the Caribbean: At World's End
  장르: Adventure Fantasy Action
  벡터 형태:
    dtype: int64
    sparse shape: (1, 22)

영화 3: Spectre
  장르: Action Adventure Crime
  벡터 형태:
    dtype: int64
    sparse shape: (1, 22)

영화 4: The Dark Knight Rises
  장르: Action Crime Drama Thriller
  벡터 형태:
    dtype: int64
    sparse shape: (1, 22)

영화 5: John Carter
  장르: Action Adventure Science Fiction
  벡터 형태:
    dtype: int64
    sparse shape: (1, 22)


---
## 5. 유사도 계산

구한 벡터를 활용하여 영화 간 유사도를 계산합니다.

In [ ]:
# cosine_similarity 로 유사도 행렬 계산
genre_vectors_dense = genre_vectors.toarray()
print("유사도 행렬 계산:")
similarity = cosine_similarity(genre_vectors_dense)

# 결과 정보
print(f"유사도 행렬의 shape: {similarity.shape}")
print("\n예시: 첫 번째 영화와 각 영화 간의 유사도")
print(f"  상위 10 개 유사도: {similarity[0].sort_values(ascending=False)[:10]}")

유사도 행렬 계산:
유사도 행렬의 shape: (4803, 4803)

예시: 첫 번째 영화와 각 영화 간의 유사도


AttributeError: 'numpy.ndarray' object has no attribute 'sorted'

---
## 6. 추천 시스템 구현

특정 영화와 유사한 영화를 top_n 개 추천하는 함수를 구현합니다.

In [ ]:
# 추천 함수 구현
def recommend_movies(id, top_n=10):
    """
    특정 영화와 유사한 영화를 top_n 개 추천합니다.
    
    Parameters:
    -----------
    id : int
        추천할 영화의 ID
    top_n : int (기본값: 10)
        추천할 영화의 개수
    
    Returns:
    --------
    list of dict
        각 영화의 id, title, similarity 점수
    """
    # id 가 movies 에 있는지 확인
    if id not in movies['id'].values:
        raise ValueError(f"id={id}를 찾지 못했습니다.")
    
    # 해당 영화의 유사도 행렬에서 한 줄 추출
    movie_similarity_scores = similarity[id]
    
    # 자기 자신 제외 (유사도 1.0)
    movie_df = pd.DataFrame({
        'id': movies['id'],
        'title': movies['title'],
        'similarity_score': similarity_scores
    })
    
    # 자기 자신 제외 및 유사도 순으로 정렬 (하위 top_n)
    recommended = movie_df.sort_values('similarity_score', ascending=False).iloc[1:top_n+1]
    
    # 결과 반환
    return recommended[['id', 'title', 'similarity_score']].to_dict(orient='records')


In [23]:
# 추천 시스템 테스트
print("추천 시스템 테스트:")

# 'Titanic' 영화의 ID 찾기
titanic_id = movies[movies['title'] == 'Titanic']['id'].values[0]
print(f"\n'Titanic'의 id: {titanic_id}")

# Titanic 의 추천 영화 (top 10)
top_10_recs = recommend_movies(titanic_id, top_n=10)

추천 시스템 테스트:

'Titanic'의 id: 597


NameError: name 'similarity_scores' is not defined

In [24]:
# 추천 결과 출력
print("\n'Titanic'과 유사한 영화 Top 10:")
for i, rec in enumerate(top_10_recs):
    print(f"{i+1}. {rec['title']} (유사도: {rec['similarity_score']:.4f})")


'Titanic'과 유사한 영화 Top 10:


NameError: name 'top_10_recs' is not defined

In [25]:
# 다른 영화로도 테스트
print("\n'Inception'의 추천:")

# 'Inception' 영화의 ID 찾기
inception_id = movies[movies['title'] == 'Inception']['id'].values[0]
print(f"'Inception'의 id: {inception_id}")

# 추천 영화 (top 5)
top_5_recs = recommend_movies(inception_id, top_n=5)

# 결과 출력
for i, rec in enumerate(top_5_recs):
    print(f"{i+1}. {rec['title']} (유사도: {rec['similarity_score']:.4f})")


'Inception'의 추천:
'Inception'의 id: 27205


IndexError: index 27205 is out of bounds for axis 0 with size 4803

---
## 요약

### 구현 완료된 기능:

1. **데이터 준비 및 이해**: tmdb_5000_movies 데이터셋 로드 및 구조 파악
2. **데이터 전처리**: genres 와 keywords 칼럼을 리스트로 변환 (eval 사용)
3. **피처 엔지니어링**: genres_literal 컬럼 생성 (장르 이름만 추출 및 공백 연결)
4. **벡터화**: CountVectorizer 를 활용한 벡터화
   - 벡터의 shape: {shape[0]} x {shape[1]}
   - 전체 데이터 개수: {len(movies)}
   - 고유 장르 수: {len(unique_genres_list)}
5. **유사도 계산**: cosine_similarity 를 활용한 영화 간 유사도 행렬 계산
6. **추천 시스템 구현**: `recommend_movies(id, top_n)` 함수 구현

**사용 방법:**
```python
# 특정 영화의 id 찾기
id = movies[movies['title'] == 'Titanic']['id'].values[0]

# 추천받기 (top 10)
recommendations = recommend_movies(id, top_n=10)
```

---
**참고:** 이 프로젝트는 TMDB 5000 Movies 데이터셋을 기반으로 한 간단한 협업 필터링 추천 시스템을 구현한 것입니다.